# EGRFC Backend Exploration

Simple notebook for interrogating the canonical backend database (`data/egrfc_backend.duckdb`).

In [1]:
from pathlib import Path
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

candidate_paths = [
    Path.cwd() / "data" / "egrfc_backend.duckdb",
    Path.cwd() / "data" / "egrfc_backend_alt.duckdb",
]

last_error = None
con = None
db_path = None

for candidate in candidate_paths:
    if not candidate.exists():
        continue
    try:
        con = duckdb.connect(str(candidate), read_only=True)
        db_path = candidate
        break
    except Exception as exc:
        last_error = exc

if con is None:
    if last_error:
        raise RuntimeError(f"Could not open any backend DB in read-only mode: {last_error}")
    raise FileNotFoundError("No backend DB found. Build one with python/build_backend.py")

print(f"Using database (read-only): {db_path}")

Using database (read-only): /Users/samlindsay/Documents/Projects/Personal/EGRFC/egrfc-stats/data/egrfc_backend.duckdb


In [2]:
# Quick helper
def q(sql: str, limit: int | None = None):
    df = con.execute(sql).df()
    if limit is not None:
        return df.head(limit)
    return df

## 1) Table row counts

In [5]:
q("""
SELECT 'games' AS table_name, COUNT(*) AS rows FROM games
UNION ALL SELECT 'player_appearances', COUNT(*) FROM player_appearances
UNION ALL SELECT 'lineouts', COUNT(*) FROM lineouts
UNION ALL SELECT 'set_piece', COUNT(*) FROM set_piece
UNION ALL SELECT 'season_scorers', COUNT(*) FROM season_scorers
UNION ALL SELECT 'players', COUNT(*) FROM players
ORDER BY table_name
""")

,table_name,rows
0,games,271
1,lineouts,1230
2,player_appearances,4996
3,players,281
4,season_scorers,962
5,set_piece,220


## 2) Season coverage

In [6]:
q("""
SELECT season, squad, COUNT(*) AS games
FROM games
GROUP BY season, squad
ORDER BY squad, season
""")

,season,squad,games
0,2016/17,1st,1
1,2017/18,1st,21
2,2018/19,1st,21
3,2019/20,1st,16
4,2021/22,1st,20
5,2022/23,1st,20
6,2023/24,1st,26
7,2024/25,1st,24
8,2025/26,1st,18
9,2016/17,2nd,6


In [11]:
q("""
SELECT season, squad, COUNT(*) AS appearances
FROM player_appearances
GROUP BY season, squad
ORDER BY squad, season
""")

,season,squad,appearances
0,2016/17,1st,19
1,2017/18,1st,324
2,2018/19,1st,350
3,2019/20,1st,270
4,2021/22,1st,399
5,2022/23,1st,364
6,2023/24,1st,491
7,2024/25,1st,436
8,2025/26,1st,334
9,2016/17,2nd,108


## 3) Integrity checks

In [7]:
# Duplicate keys check
checks = {
    "games_pk_dupes": """
        SELECT squad, date, opposition, COUNT(*) AS c
        FROM games
        GROUP BY squad, date, opposition
        HAVING COUNT(*) > 1
    """,
    "apps_pk_dupes": """
        SELECT squad, date, player, COUNT(*) AS c
        FROM player_appearances
        GROUP BY squad, date, player
        HAVING COUNT(*) > 1
    """,
    "lineouts_pk_dupes": """
        SELECT squad, date, seq_id, COUNT(*) AS c
        FROM lineouts
        GROUP BY squad, date, seq_id
        HAVING COUNT(*) > 1
    """,
}

for name, sql in checks.items():
    df = q(sql)
    print(f"{name}: {len(df)} issue rows")
    if not df.empty:
        display(df.head(20))

games_pk_dupes: 0 issue rows
apps_pk_dupes: 0 issue rows
lineouts_pk_dupes: 0 issue rows


In [8]:
# Sanity checks
q("""
SELECT
  SUM(CASE WHEN score_for IS NULL OR score_against IS NULL THEN 1 ELSE 0 END) AS games_missing_score,
  SUM(CASE WHEN lineouts_total < lineouts_won THEN 1 ELSE 0 END) AS lineout_bad_totals,
  SUM(CASE WHEN scrums_total < scrums_won THEN 1 ELSE 0 END) AS scrum_bad_totals
FROM games
LEFT JOIN set_piece USING (game_id)
""")

,games_missing_score,lineout_bad_totals,scrum_bad_totals
0,0.0,0.0,0.0


## 4) View outputs used by frontend/charts

In [9]:
q("SELECT * FROM v_season_results ORDER BY season DESC, squad, game_type")

,season,squad,game_type,played,won,lost,drawn,points_for,points_against
0,2025/26,1st,Cup,3,2.0,1.0,0.0,94.0,77.0
1,2025/26,1st,Friendly,1,0.0,1.0,0.0,19.0,24.0
2,2025/26,1st,League,14,12.0,2.0,0.0,512.0,191.0
3,2025/26,2nd,Cup,2,1.0,1.0,0.0,63.0,86.0
4,2025/26,2nd,League,9,5.0,4.0,0.0,177.0,199.0
5,2024/25,1st,Cup,2,1.0,1.0,0.0,61.0,65.0
6,2024/25,1st,League,22,3.0,18.0,1.0,380.0,717.0
7,2024/25,2nd,Cup,1,0.0,1.0,0.0,22.0,31.0
8,2024/25,2nd,Friendly,6,4.0,2.0,0.0,154.0,104.0
9,2024/25,2nd,League,13,7.0,6.0,0.0,317.0,308.0


In [10]:
q("SELECT * FROM v_set_piece_summary ORDER BY season DESC, squad, team")

,squad,season,team,avg_lineouts_success_rate,avg_scrums_success_rate,avg_points_per_22m_entry,avg_tries_per_22m_entry,games
0,1st,2025/26,EGRFC,0.821544,0.978205,NaN,NaN,13
1,1st,2025/26,Opposition,0.649246,0.692578,NaN,NaN,13
2,2nd,2025/26,EGRFC,0.777679,0.858730,NaN,NaN,4
3,2nd,2025/26,Opposition,0.758333,0.887581,NaN,NaN,4
4,1st,2024/25,EGRFC,0.714763,0.976431,NaN,NaN,18
5,1st,2024/25,Opposition,0.795167,0.787650,NaN,NaN,18
6,2nd,2024/25,EGRFC,0.670615,0.843074,NaN,NaN,11
7,2nd,2024/25,Opposition,0.766863,0.909848,NaN,NaN,11
8,1st,2023/24,EGRFC,0.698811,0.917893,NaN,NaN,21
9,1st,2023/24,Opposition,0.798663,0.781265,NaN,NaN,21


In [186]:
# Lineout success by area
zone = q(
    """
    SELECT 
        CASE WHEN area == 'Front' THEN 1 WHEN area == 'Middle' THEN 2 WHEN area == 'Back' THEN 3 END AS zone, 
        SUM(total) AS total, 
        SUM(won) AS won, 
        ROUND(SUM(won) * 100.0 / NULLIF(SUM(total), 0), 1) / 100.0 AS pct_won 
        FROM v_lineout_summary 
        GROUP BY zone 
        ORDER BY pct_won DESC
    """, 
    limit=100)

jumper_sql = """
    SELECT 
        'Jumper' AS player_type,
        jumper AS player, 
        SUM(total) AS total, 
        SUM(won) AS won, 
        ROUND(SUM(won) * 100.0 / NULLIF(SUM(total), 0), 1) / 100.0 AS pct_won, 
        SQRT((SUM(won)/SUM(total) * (1 - SUM(won)/SUM(total)))/SUM(total)) AS error_margin,
        SUM(total * zone)/SUM(total) AS avg_zone
    FROM (SELECT *, CASE WHEN area == 'Front' THEN 1 WHEN area == 'Middle' THEN 2 WHEN area == 'Back' THEN 3 END AS zone FROM v_lineout_summary) AS subquery
    WHERE squad = '1st'
    GROUP BY jumper
    -- HAVING SUM(total) >= 20
    ORDER BY pct_won DESC
    """

thrower_sql = """
    SELECT 
        'Thrower' AS player_type,
        thrower AS player, 
        SUM(total) AS total, 
        SUM(won) AS won, 
        ROUND(SUM(won) * 100.0 / NULLIF(SUM(total), 0), 1) / 100.0 AS pct_won, 
        SQRT((SUM(won)/SUM(total) * (1 - SUM(won)/SUM(total)))/SUM(total)) AS error_margin,
        SUM(total * zone)/SUM(total) AS avg_zone
    FROM (SELECT *, CASE WHEN area == 'Front' THEN 1 WHEN area == 'Middle' THEN 2 WHEN area == 'Back' THEN 3 END AS zone FROM v_lineout_summary) AS subquery
    WHERE squad = '1st'
    GROUP BY thrower
    --HAVING SUM(total) >= 20
    ORDER BY pct_won DESC
    """

jumpers = q(jumper_sql, limit=100)

throwers = q(thrower_sql, limit=100)

players = q(f"({jumper_sql}) UNION ALL ({thrower_sql})")

players

,player_type,player,total,won,pct_won,error_margin,avg_zone
0,Jumper,Ryan McGealy,2.0,2.0,1.000,0.000000,1.000000
1,Jumper,Or Shay,3.0,3.0,1.000,0.000000,1.000000
2,Jumper,Will Bramwell,4.0,4.0,1.000,0.000000,1.250000
3,Jumper,Guy Collins,1.0,1.0,1.000,0.000000,1.000000
4,Jumper,Josh Brimecombe,1.0,1.0,1.000,0.000000,1.000000
5,Jumper,Oscar Staples,21.0,19.0,0.905,0.064056,1.809524
6,Jumper,Rory Evans,39.0,34.0,0.872,0.053534,1.589744
7,Jumper,John Peaty,227.0,189.0,0.833,0.024779,1.546256
8,Jumper,Pete Morley,12.0,10.0,0.833,0.107583,1.583333
9,Jumper,Sam Lindsay-McCall,388.0,303.0,0.781,0.020998,2.244845


In [187]:

# Plot success (y-axis) line by zone (x-axis), with scatter points for jumpers by average zone
import altair as alt
from python.chart_helpers import *

size_scale = alt.Scale(domain=[10,1000], range=[10,400])

base = alt.Chart(zone).encode(
    x=alt.X(
        "zone:Q", 
        title="Average Lineout Zone", 
        axis=alt.Axis(tickCount=3, values=[1, 2, 3], labelExpr="datum.value == 1 ? 'Front' : datum.value == 2 ? 'Middle' : 'Back'", grid=False, ticks=False, labelFontSize=16), 
        scale=alt.Scale(domain=[0.75, 3.25], nice=False)
    ),
    y=alt.Y("pct_won:Q", title="Success Rate", scale=alt.Scale(domain=[0.5, 1.0]), axis=alt.Axis(format="%", tickCount=5)),
)   

avg_points = base.encode(size=alt.Size("total:Q", legend=None, scale=size_scale)).mark_point(color="#991515", fill="red", fillOpacity=0.9)
avg_line = base.mark_line(color="#991515")
avg_total_labels = avg_points.encode(text="total:Q", size=alt.value(12)).mark_text(align="right", dx=-10, dy=5, opacity=0.5, color="#991515")

points = alt.Chart(players).encode(
    x=alt.X("avg_zone:Q", title="Average Lineout Zone"),
    y=alt.Y("pct_won:Q", title="Success Rate"),
    size=alt.Size("total:Q", legend=None, scale=size_scale),
    color=alt.Color("player_type:N", scale=alt.Scale(domain=["Thrower", "Jumper"], range=["#146f14","#202946"]), title=None,legend=alt.Legend(orient="top-right")),
    tooltip=["player_type", "player", "total", "won", "pct_won", "avg_zone"],
)

errorbars = points.mark_errorbar(opacity=0.5).encode(
    x="avg_zone:Q",
    y=alt.Y("ymin:Q", scale=alt.Scale(clamp=True), title="Success Rate"),
    y2=alt.Y2("ymax:Q", title=None)
).transform_calculate(
    ymin="datum.pct_won - datum.error_margin",
    ymax="datum.pct_won + datum.error_margin"
)

# Add text labels for jumpers
labels = points.encode(text="player:N", size=alt.value(12)).mark_text(align="left", dx=10)
total_labels = points.encode(
    text="total:Q",
    size=alt.value(12), 
    color=alt.value("black")
).mark_text(align="right", dx=-10, opacity=0.5)

chart = (
    avg_line
    + avg_points
    + avg_total_labels
    + errorbars 
    + points.mark_circle(size=100, opacity=1., fillOpacity=0.9) 
    + labels
    + total_labels
).resolve_scale(
    y="shared", x="shared"
).properties(
    title=alt.Title(
        text="Lineout Success by Zone", 
        subtitle=[
            "1st XV average success rate since 2021 (red), with individual players by their average zone.",
            "Size and numbers indicate total lineouts taken. Error bars show uncertainty based on sample size."
        ]
    ),
    width=500,
    height=600
)
chart.save("lineout_success_by_zone.html")

## 5) Player deep dive

In [12]:
player_name = "Jake Radcliffe"  # change as needed

print("Profile")
display(q(f"SELECT * FROM players WHERE name = '{player_name}'"))

print("Appearances by season")
display(q(f"""
    SELECT season, squad, COUNT(*) AS apps, SUM(CASE WHEN is_starter THEN 1 ELSE 0 END) AS starts
    FROM player_appearances
    WHERE player = '{player_name}'
    GROUP BY season, squad
    ORDER BY squad, season
"""))

print("Scoring by season")
display(q(f"""
    SELECT season, squad, tries, conversions, penalties, drop_goals, points
    FROM season_scorers
    WHERE player = '{player_name}'
    ORDER BY squad, season
"""))

print("Selections by position")
display(q(f"""
    SELECT position, COUNT(*) AS selections
    FROM player_appearances
    WHERE player = '{player_name}'
    GROUP BY position
    ORDER BY position
"""))

Profile


,name,short_name,position,squad,first_appearance_date,first_appearance_squad,first_appearance_opposition,photo_url,sponsor,total_appearances,total_starts,total_captaincies,total_vc_appointments,total_lineouts_jumped,lineouts_won_as_jumper,career_points
0,Jake Radcliffe,Jake R,Wing,Both,2017-09-23,1st,Barns Green,img/headshots/JakeRadcliffe.png,EGRFC Touch Rugby,100,89,3,5,0,0,340


Appearances by season


,season,squad,apps,starts
0,2017/18,1st,7,4.0
1,2018/19,1st,3,1.0
2,2019/20,1st,4,4.0
3,2021/22,1st,15,14.0
4,2022/23,1st,10,10.0
5,2023/24,1st,18,18.0
6,2024/25,1st,19,18.0
7,2025/26,1st,15,14.0
8,2017/18,2nd,5,5.0
9,2019/20,2nd,1,1.0


Scoring by season


,season,squad,tries,conversions,penalties,drop_goals,points
0,2017/18,1st,0,0,0,0,0
1,2018/19,1st,1,1,0,0,7
2,2019/20,1st,6,0,0,0,30
3,2021/22,1st,11,0,0,0,55
4,2022/23,1st,12,0,0,0,60
5,2023/24,1st,11,0,0,0,55
6,2024/25,1st,6,0,0,0,30
7,2025/26,1st,10,3,0,0,56
8,2017/18,2nd,7,1,0,0,37
9,2018/19,2nd,1,1,1,0,10


Selections by position


,position,selections
0,Bench,9
1,Centre,18
2,Full Back,3
3,Wing,68


## 6) Team/opposition drilldown

In [16]:
opposition = "Hove"  # change as needed
q(f"""
SELECT date, season, squad, home_away, score_for, score_against, result, game_type
FROM games
WHERE opposition = '{opposition}'
ORDER BY date DESC
""")

,date,season,squad,home_away,score_for,score_against,result,game_type
0,2026-02-14,2025/26,2nd,H,7,47,L,League
1,2025-11-15,2025/26,1st,H,15,32,L,Cup
2,2025-11-08,2025/26,2nd,A,19,59,L,League
3,2025-02-08,2024/25,1st,A,11,18,L,League
4,2024-10-12,2024/25,1st,H,21,34,L,League
5,2024-03-23,2023/24,1st,H,13,15,L,League
6,2023-12-09,2023/24,1st,A,26,12,W,League
7,2023-03-04,2022/23,2nd,A,24,57,L,League
8,2023-01-07,2022/23,2nd,H,43,19,W,League
9,2022-05-07,2021/22,2nd,A,29,20,W,Cup


In [ ]:
# Close connection when done
con.close()